# Face Anonymization for Photogrammetry Scans

GDPR-compliant face removal pipeline for Einstar 3D photogrammetry scans.
Removes facial geometry while preserving all anatomical landmarks and optode positions.

**Pipeline:**
1. Load scan and detect nasion (auto via MediaPipe, or manual click)
2. Normalize axes, isolate head, detect landmarks, compute face mask + ear coverage
3. Preview the detected region
4. Interactive refinement: click on missed spots to add circular deletion zones
5. Delete facial vertices and visualize result

In [ ]:
import logging

import numpy as np
import pyvista as pv
import cedalion
import cedalion.io
import cedalion.plots
from cedalion.geometry.photogrammetry.anonymization import (
    detect_nasion_auto,
    pick_nasion,
    isolate_head,
    eye_plane_rotation_matrix,
    heuristic_lpa_rpa,
    mediapipe_face_mask_from_contour,
    refine_mask_interactive,
    delete_masked_vertices,
)
from cedalion.vtktutils import trimesh_to_vtk_polydata

# Suppress verbose module and MediaPipe logging
logging.getLogger('cedalion').setLevel(logging.WARNING)
logging.getLogger('absl').setLevel(logging.WARNING)

pv.set_jupyter_backend('server')

# === CONFIG ===
SUBJECT_NUMBER = 21
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'
FORCE_MANUAL = False
CIRCLE_RADIUS = 25.0       # mm -- brush radius for interactive refinement


## 1. Load scan + detect nasion

Auto-detect nasion via MediaPipe. If no face is found (or `FORCE_MANUAL=True`),
falls back to an interactive picker where the user clicks the nasion.

In [2]:
path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
surface = cedalion.io.read_einstar_obj(path)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# Nasion detection
if not FORCE_MANUAL:
    auto_result = detect_nasion_auto(surface)
else:
    auto_result = None

if auto_result is not None:
    nasion, meta = auto_result
    print(f'Auto nasion: method={meta["method"]}, confidence={meta["confidence"]:.2f}')
else:
    print('Opening manual nasion picker...')
    nasion = pick_nasion(surface)
    if nasion is None:
        raise RuntimeError('No nasion selected — cannot proceed')
    meta = {'method': 'manual', 'confidence': 1.0}
    print(f'Manual nasion at: [{nasion[0]:.1f}, {nasion[1]:.1f}, {nasion[2]:.1f}]')

fwd = meta.get('forward_direction')
face_contour = meta.get('face_contour_3d')
eyes = meta.get('eyes')
print(f'Forward direction: {"available" if fwd is not None else "N/A"}')
print(f'Face contour: {len(face_contour)} pts' if face_contour is not None else 'Face contour: N/A')
print(f'Eyes: {"available" if eyes is not None else "N/A"}')

Loaded: 494,109 vertices, 866,676 faces


W0000 00:00:1776065866.226577  601547 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776065866.263782  601547 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776065866.277111  601573 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.4-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, ACO, DRM 3.64, 6.19.11-arch1-1)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776065866.283648  601557 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776065866.294381  601555 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Auto nasion: method=mediapipe+unified, confidence=0.90
Forward direction: available
Face contour: 36 pts
Eyes: available


## 2. Eye-plane rotation, head isolation, heuristic LPA/RPA, face mask


In [ ]:
# --- Eye-plane rotation (replaces normalize_axes) ---
# Eyes come from the same validated sweep as the nasion (see nasion_detector).
if eyes is None:
    raise RuntimeError(
        "No eyes in nasion metadata -- unified sweep failed. "
        "Fall back to manual pipeline."
    )
r_eye_raw, l_eye_raw = eyes
R_eye = eye_plane_rotation_matrix(r_eye_raw, l_eye_raw, fwd)

# Rotate mesh about its centroid
rotated_mesh = surface.mesh.copy()
raw_verts = np.asarray(rotated_mesh.vertices, dtype=float)
rot_center = raw_verts.mean(axis=0)
rotated_mesh.vertices = (R_eye @ (raw_verts - rot_center).T).T + rot_center
surface_n = cedalion.dataclasses.TrimeshSurface(
    mesh=rotated_mesh, crs=surface.crs, units=surface.units
)
nasion_n = R_eye @ (np.asarray(nasion) - rot_center) + rot_center
r_eye_n  = R_eye @ (r_eye_raw - rot_center) + rot_center
l_eye_n  = R_eye @ (l_eye_raw - rot_center) + rot_center
print(f'Eye-plane rotation applied. Canonical frame: X=up, Y=forward, Z=left')

# --- Isolate head (library call strips disconnected fragments automatically) ---
surface_h, _ = isolate_head(surface_n, nasion_n)
print(f'Head isolation: {surface_n.nvertices:,} -> {surface_h.nvertices:,} vertices')

# --- Heuristic LPA/RPA in the eye-plane canonical frame ---
verts = np.asarray(surface_h.mesh.vertices)
head_centroid = verts.mean(axis=0)
eye_x = 0.5 * (r_eye_n[0] + l_eye_n[0])
lpa_heur, rpa_heur = heuristic_lpa_rpa(verts, eye_x, head_centroid[1])
if lpa_heur is None or rpa_heur is None:
    raise RuntimeError('heuristic_lpa_rpa returned None -- ear slice had < 10 candidates')
lpa_c, lpa_r = lpa_heur['center'], lpa_heur['radius']
rpa_c, rpa_r = rpa_heur['center'], rpa_heur['radius']

# --- Rotate MediaPipe face oval into the canonical frame (if available) ---
if face_contour is not None:
    contour_n = (R_eye @ (np.asarray(face_contour) - rot_center).T).T + rot_center
else:
    contour_n = None

# --- Build deletion mask ---
extended_mask, mask_info = mediapipe_face_mask_from_contour(
    verts, nasion_n, lpa_c, lpa_r, rpa_c, rpa_r,
    face_contour_rotated=contour_n,
)
print(f'Mask: {extended_mask.sum():,} vertices '
      f'({100 * extended_mask.sum() / len(extended_mask):.1f}%)')
for k, v in mask_info['counts'].items():
    print(f'  {k:24s} {v:,}')
print(f'  forehead upper bound: {mask_info["arc_source"]}')

protected_positions = np.stack([nasion_n, lpa_c, rpa_c])
protected_labels = ['Nz', 'LPA_heur', 'RPA_heur']


## 3. Preview anonymization region

In [4]:
pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface_h, opacity=1.0)

# Deletion region (red scatter)
if extended_mask.sum() > 0:
    pvplt.add_mesh(
        pv.PolyData(verts[extended_mask]), color="red", point_size=3,
        opacity=0.4, render_points_as_spheres=True,
        label=f"deletion region ({extended_mask.sum():,})",
    )

# Heuristic LPA/RPA as translucent spheres (NOT to be deleted)
pvplt.add_mesh(pv.Sphere(radius=lpa_r, center=lpa_c), color="orange", opacity=0.35)
pvplt.add_point_labels([lpa_c], ["LPA heur"], font_size=14,
                       text_color="orange", bold=True, shape=None)
pvplt.add_mesh(pv.Sphere(radius=rpa_r, center=rpa_c), color="blue", opacity=0.35)
pvplt.add_point_labels([rpa_c], ["RPA heur"], font_size=14,
                       text_color="blue", bold=True, shape=None)

# Nasion
pvplt.add_mesh(pv.Sphere(radius=3, center=nasion_n), color="lime")
pvplt.add_point_labels([nasion_n], ["Nz"], font_size=16,
                       text_color="lime", shape=None, always_visible=True)

# Eye positions (for reference)
pvplt.add_mesh(pv.Sphere(radius=3, center=r_eye_n), color="cyan")
pvplt.add_mesh(pv.Sphere(radius=3, center=l_eye_n), color="cyan")

# MediaPipe face oval (orange points) -- rotate into canonical frame
if face_contour is not None:
    contour_n = (R_eye @ (np.asarray(face_contour) - rot_center).T).T + rot_center
    pvplt.add_mesh(
        pv.PolyData(contour_n), color="orange", point_size=14,
        render_points_as_spheres=True,
        label=f"MP face oval ({len(contour_n)} pts)",
    )

pvplt.add_text(
    f"S{SUBJECT_NUMBER} | deletion region: {extended_mask.sum():,} vertices",
    position="upper_left", font_size=12,
)
pvplt.add_legend()
pvplt.show()


Widget(value='<iframe src="http://localhost:40923/index.html?ui=P_0x7fc9d27b2350_0&reconnect=auto" class="pyvi…

## 4. Interactive refinement — click to add or remove deletion zones

- **Left-click**: Add/remove vertices within the brush radius
- **T**: Toggle between **add** mode (mark for deletion) and **remove** mode (undo)
- **+** / **-**: Increase / decrease brush radius
- Close the window when done.

In [ ]:
# Interactive click-to-paint refinement (library call).
extended_mask = refine_mask_interactive(
    surface_h,
    extended_mask,
    protected_positions,
    protected_labels=protected_labels,
    brush_radius=CIRCLE_RADIUS,
)
print(f'Refined mask: {extended_mask.sum():,} vertices')


## 5. Anonymize — delete facial vertices

In [ ]:
# Delete faces touching any masked vertex (library call).
anonymized_surface = delete_masked_vertices(surface_h, extended_mask)
n_removed = surface_h.nvertices - anonymized_surface.nvertices
print(f'Original:    {surface_h.nvertices:,} verts, {surface_h.nfaces:,} faces')
print(f'Anonymized:  {anonymized_surface.nvertices:,} verts, {anonymized_surface.nfaces:,} faces')
print(f'Removed:     {n_removed:,} verts')

# Side-by-side visualization
anon_vtk_mesh = trimesh_to_vtk_polydata(anonymized_surface.mesh)

pvplt = pv.Plotter(shape=(1, 2))
pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface_h, opacity=1.0)
pvplt.add_text(f'S{SUBJECT_NUMBER} Original', position='upper_left', font_size=14)
pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk_mesh), rgb=True)
pvplt.add_text(f'S{SUBJECT_NUMBER} Anonymized (-{n_removed:,} verts)',
               position='upper_left', font_size=14)
pvplt.link_views()
pvplt.show()


## 6. Anonymized head — original color

In [8]:
pvplt = pv.Plotter()
pvplt.add_mesh(pv.wrap(anon_vtk_mesh), rgb=True, smooth_shading=True)
pvplt.add_text(f'S{SUBJECT_NUMBER} Anonymized', position='upper_left', font_size=14)
pvplt.show()

Widget(value='<iframe src="http://localhost:42885/index.html?ui=P_0x7f8fe792ab90_2&reconnect=auto" class="pyvi…